# RK1828 Qwen3-VL 视觉语言大模型 Demo


[Qwen3-VL](https://github.com/QwenLM/Qwen3-VL)（通义千问）视觉语言模型：
看图回答 / 图像描述。LLM 部分 **GRQ W4A16 量化**，视觉编码器 fp16，全部跑在 RK1828 上。

| 规格 | 生成速度（实测） |
|---|---|
| Qwen3-VL-2B | 62.7 tokens/s |
| Qwen3-VL-4B | 38.7 tokens/s |

| 步骤 | 内容 |
|---|---|
| 1 | 下载模型（可选 HF / HF-Mirror 源，可只下 2B 或 4B） |
| 2 | 解压预编译 demo |
| 3 | 输入提示词，看图说话 |

## 0. 环境检查

先确认 RKNN3 运行库（`librknn3_api.so`）和 RK1828 设备都正常。
如失败，请先安装驱动/运行时（`rknn3-rk182x` DEB 包），并用 `sudo rknn-smi info` 确认能看到 RK1828。

In [ ]:
from rknn3 import get_lib, find_devices

_ = get_lib()  # 加载 /usr/lib/librknn3_api.so；失败说明 Runtime 未安装

devices = find_devices()
if not devices:
    raise RuntimeError("未发现 RK1828 设备！请检查: sudo rknn-smi info")
for dev_id, dev_type in devices:
    print(f"发现设备: id={dev_id}  接口={dev_type}")

## 1. 下载模型

本 demo 的模型文件托管在 Hugging Face（`qhddz` 账号）。
**先用下拉框选择下载源**，再运行下一个单元格开始下载：

- **HF-Mirror 国内镜像**：`hf-mirror.com`，国内直连，推荐
- **Hugging Face 官方**：`huggingface.co`，海外或已有代理时用

> 已存在（且非空）的文件会**自动跳过**，重复运行不会重新下载。


两个规格共约 10GB。磁盘紧张的话，下载单元格可以改成只下载其中一个规格。

In [ ]:
from nb_download import source_picker

# (HF仓库名, 仓库内路径, 本地路径)
FILES = [
    ("qwen3-vl-2b-rknn3-rk1828", "Qwen3-VL-2B-llm.embed.bin", "demos/qwen3_vl_2b/Qwen3-VL-2B-llm.embed.bin"),
    ("qwen3-vl-2b-rknn3-rk1828", "Qwen3-VL-2B-llm.rknn", "demos/qwen3_vl_2b/Qwen3-VL-2B-llm.rknn"),
    ("qwen3-vl-2b-rknn3-rk1828", "Qwen3-VL-2B-llm.tokenizer.gguf", "demos/qwen3_vl_2b/Qwen3-VL-2B-llm.tokenizer.gguf"),
    ("qwen3-vl-2b-rknn3-rk1828", "Qwen3-VL-2B-llm.weight", "demos/qwen3_vl_2b/Qwen3-VL-2B-llm.weight"),
    ("qwen3-vl-2b-rknn3-rk1828", "Qwen3-VL-2B-vision.rknn", "demos/qwen3_vl_2b/Qwen3-VL-2B-vision.rknn"),
    ("qwen3-vl-2b-rknn3-rk1828", "Qwen3-VL-2B-vision.weight", "demos/qwen3_vl_2b/Qwen3-VL-2B-vision.weight"),
    ("qwen3-vl-2b-rknn3-rk1828", "demo_x86_64.tar.gz", "demos/qwen3_vl_2b/demo_x86_64.tar.gz"),
    ("qwen3-vl-2b-rknn3-rk1828", "model/demo.jpg", "demos/qwen3_vl_2b/model/demo.jpg"),
    ("qwen3-vl-4b-rknn3-rk1828", "Qwen3-VL-4B-llm.embed.bin", "demos/qwen3_vl_4b/Qwen3-VL-4B-llm.embed.bin"),
    ("qwen3-vl-4b-rknn3-rk1828", "Qwen3-VL-4B-llm.rknn", "demos/qwen3_vl_4b/Qwen3-VL-4B-llm.rknn"),
    ("qwen3-vl-4b-rknn3-rk1828", "Qwen3-VL-4B-llm.tokenizer.gguf", "demos/qwen3_vl_4b/Qwen3-VL-4B-llm.tokenizer.gguf"),
    ("qwen3-vl-4b-rknn3-rk1828", "Qwen3-VL-4B-llm.weight", "demos/qwen3_vl_4b/Qwen3-VL-4B-llm.weight"),
    ("qwen3-vl-4b-rknn3-rk1828", "Qwen3-VL-4B-vision.rknn", "demos/qwen3_vl_4b/Qwen3-VL-4B-vision.rknn"),
    ("qwen3-vl-4b-rknn3-rk1828", "Qwen3-VL-4B-vision.weight", "demos/qwen3_vl_4b/Qwen3-VL-4B-vision.weight"),
    ("qwen3-vl-4b-rknn3-rk1828", "demo_x86_64.tar.gz", "demos/qwen3_vl_4b/demo_x86_64.tar.gz"),
    ("qwen3-vl-4b-rknn3-rk1828", "model/demo.jpg", "demos/qwen3_vl_4b/model/demo.jpg"),
]

w_source = source_picker()

In [ ]:
from nb_download import ensure_files

ensure_files(FILES, w_source.value)

## 2. 解压预编译 demo

`demo_x86_64.tar.gz` 里是官方流程编译好的 x86_64 C++ demo 和运行时库（`lib/`）。
HF 不允许直接托管可执行文件，所以打包成 tar.gz，解压一次即可（重复运行自动跳过）。

> **为什么这个 demo 用 C++ 而不是纯 Python？**
> 01-05（分类/检测/FastSAM/pose/分割）是单个模型一次前向，后处理简单，
> 所以用 Python（`rknn3.py`）逐步演示。而本 demo 的流水线复杂得多——
> 多模型串联、tokenizer、KV cache、滑窗/流式状态等，前后处理和编排逻辑
> 有几百上千行，重写 Python 版成本高且容易和官方实现不一致。
> 因此直接运行官方 `rknn3-model-zoo` 编译的 C++ demo，
> 源码见 [github.com/airockchip/rknn3-model-zoo](https://github.com/airockchip/rknn3-model-zoo)
> 的 `examples/` 目录。

In [ ]:
from nb_download import extract_tarball

extract_tarball("demos/qwen3_vl_2b/demo_x86_64.tar.gz", "demos/qwen3_vl_2b")
extract_tarball("demos/qwen3_vl_4b/demo_x86_64.tar.gz", "demos/qwen3_vl_4b")

In [ ]:
from nb_demo import run_demo, snapshot, new_files, show_images, show_new_images, show_audio

## 3. 选择模型规格和提示词

In [ ]:
import ipywidgets as widgets
from IPython.display import display

w_model = widgets.Dropdown(options=[("Qwen3-VL-2B（快）", "2b"),
                                    ("Qwen3-VL-4B（更聪明）", "4b")],
                           value="2b", description="模型:")
w_prompt = widgets.Textarea(value="Describe this image",
                            description="提示词:",
                            layout=widgets.Layout(width="70%", height="60px"))
display(w_model, w_prompt)
print("选好后继续往下运行 ▼")

In [ ]:
DEMO = f"demos/qwen3_vl_{w_model.value}"
tag = "Qwen3-VL-2B" if w_model.value == "2b" else "Qwen3-VL-4B"
show_images([f"{DEMO}/model/demo.jpg"], max_width=400)
run_demo(["./rknn_qwen3_vl_demo",
          f"{tag}-vision.rknn", f"{tag}-vision.weight",
          f"{tag}-llm.rknn", f"{tag}-llm.weight",
          f"{tag}-llm.tokenizer.gguf", f"{tag}-llm.embed.bin",
          "0xff", "0xff", "model/demo.jpg", w_prompt.value, "384", "384"], DEMO)

## 小结

- VLM = 视觉编码器（把图片转成 token 嵌入）+ LLM（自回归生成文字），两部分都是独立 RKNN 模型。
- `0xff 0xff` 是设备/会话参数占位；`384 384` 是视觉输入分辨率。
- 换提示词试试中文提问、数数图里有几个人、识别文字等。